#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [55]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_4.5mm.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [56]:
def guardar_cajas_y_productos(grosor):
    caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")
    cajas = [
        Caja(
            caja_id = row["caja_tipo_id"],
            dim_interior_ancho = row["caja_interior_ancho"],
            dim_interior_largo = row["caja_interior_largo"],
            dim_interior_alto = row["caja_interior_alto"]
        )
        for _, row in caja_compras_merge.iterrows()
    ]

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    # Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [57]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)

Armamos una función general para buscar un tipo de caja por su id.

In [58]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

In [59]:
solucion_base = Solucion(grosor=4.5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto,caja)
        solucion_base.agregar_asignacion(asignacion, descuentos=False)

#solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,71,02cf77de65b70dd77905e2e33d78478f,0.385062,0.983137,0.0,0.0,0.0,0.0,0.0,1005298.45,85924,12888600,13893898.45
1,BR0001,1546613,71,0835ff365412a67b720a19713ec250f3,0.928963,1.089928,0.0,0.0,0.0,0.0,0.0,1005298.45,32223,4833450,5838748.45
2,BR0001,1546613,71,125888817c4da192ce9335454b8d4432,0.875156,1.158994,0.0,0.0,0.0,0.0,0.0,1005298.45,32223,4833450,5838748.45
3,BR0001,1546613,71,170f5916eaeb0e271ae012eedb1ad645,0.828083,1.016779,0.0,0.0,0.0,0.0,0.0,1005298.45,38667,5800050,6805348.45
4,BR0001,1546613,71,1fe6b9010b30d1d429fc0c7a240159aa,0.338855,1.121933,0.0,0.0,0.0,0.0,0.0,1005298.45,85924,12888600,13893898.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23951,BR0421,145803,36,e70cd62d04659cbda3f1df24c7140e35,0.754876,1.222194,0.0,0.0,0.0,0.0,0.0,94771.95,2279,341850,436621.95
23952,BR0421,145803,36,f95687311339e4b9b0ef5622b26fd941,0.428443,1.071770,0.0,0.0,0.0,0.0,0.0,94771.95,4557,683550,778321.95
23953,BR0421,145803,36,fa8da38322149b23e14a120e71d38975,0.419599,0.953191,0.0,0.0,0.0,0.0,0.0,94771.95,5208,781200,875971.95
23954,BR0421,145803,36,fc6931875b8384c7bacd18bb4699795f,0.392084,1.022831,0.0,0.0,0.0,0.0,0.0,94771.95,5208,781200,875971.95


#### **Greedy 1: Elección por menor costo**

In [60]:
def solucion_greedy(grosor, cajas, productos_ordenados, criterio):

    solucion = Solucion(grosor=grosor)

    for producto in productos_ordenados:
        metricas = {}  # caja_id -> (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
        
        for caja_id in producto.cajas_asignables:
            caja = buscar_caja_por_id(caja_id)
            
            # Volumen requerido actual (antes de asignar este producto)
            volumen_total = caja.unidades_total_requeridas()
            utilizacion_pallet = caja.utilizacion_pallet()
            
            # Costo simulado de asignar este producto a esta caja
            asignacion = Asignacion(producto, caja)
            caja.asignar_producto(producto)
            
            # Calculamos los costos del producto utilizando ese tipo de caja
            costo_packaging = asignacion.costo_packaging_producto_total()
            costo_flete = 150 * asignacion.cant_pallets_requeridas()
            costo_total = costo_packaging + costo_flete
            metricas[caja_id] = (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
            
            # Revertimos la asignación para que no se aplique un descuento posterior
            caja.revocar_producto(producto)
        
        if criterio == "minimizar_costo_total":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][3]))
        elif criterio == "maximizar_volumen_tipo":        
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][3]))
        elif criterio == "minimizar_costo_flete":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][2]))
        elif criterio == "maximizar_utilizacion_pallet":
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][4]))
        
        caja_optima = buscar_caja_por_id(caja_id_optima)
        
        asignacion = Asignacion(producto, caja_optima)
        solucion.agregar_asignacion(asignacion)    
    
    return solucion

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [61]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy1 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()
solucion_greedy1.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 21
Costo packaging: 28858399.70000001
Costo flete: 135633450
Costo total: 164491849.70000002
Utilización de pallet promedio: 0.8854263532761351
Utilización de caja promedio: 1.2621787184573823


In [62]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy2 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy2.resumen_por_asignacion()
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 25
Costo packaging: 29067442.105000023
Costo flete: 135621450
Costo total: 164688892.10500002
Utilización de pallet promedio: 0.8831659655108764
Utilización de caja promedio: 1.2658219045897858


In [63]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy3 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy3.resumen_por_asignacion()
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 11
Costo packaging: 28679836.25
Costo flete: 155208000
Costo total: 183887836.25
Utilización de pallet promedio: 0.9134120310240889
Utilización de caja promedio: 1.0691205378647197


In [64]:
solucion_greedy3.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,71,2060590a59a5c59ff1bb32dbffe1028b,0.961331,1.052083,-0.3,-0.3,-0.3,-0.1,-0.2,715185.835,32223,4833450,5548635.835
1,BR0002,139211,76,5229dfb764d843b66341f7ca5cdec3dd,0.973841,1.271230,-0.3,-0.3,-0.3,-0.3,-0.3,63341.005,2176,326400,389741.005
2,BR0003,172506,36,67894c5a8355ca9b7e57b96362fa45f1,0.971657,0.970414,-0.2,-0.3,-0.2,-0.2,-0.2,89703.120,2157,323550,413253.120
3,BR0004,271715,80,5229dfb764d843b66341f7ca5cdec3dd,0.973841,1.095214,-0.3,-0.3,-0.3,-0.3,-0.3,123630.325,4246,636900,760530.325
4,BR0005,7586,103,5229dfb764d843b66341f7ca5cdec3dd,0.973841,1.167981,-0.3,-0.3,-0.3,-0.3,-0.3,3451.630,119,17850,21301.630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,46,35174969e008414733b7133a380c2f8c,0.852133,0.928834,-0.3,-0.3,-0.3,-0.3,-0.3,1256.255,50,7500,8756.255
423,BR0418,3942,46,35174969e008414733b7133a380c2f8c,0.852133,0.928834,-0.3,-0.3,-0.3,-0.3,-0.3,1793.610,71,10650,12443.610
424,BR0419,43300,48,67894c5a8355ca9b7e57b96362fa45f1,0.971657,1.071085,-0.2,-0.3,-0.2,-0.2,-0.2,22516.000,542,81300,103816.000
425,BR0420,205852,24,1c01c33ced1f5bfdb36a399ac98e0c68,0.832071,1.005770,-0.2,-0.3,-0.3,-0.3,-0.2,93662.660,3217,482550,576212.660


In [65]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy4 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy4.resumen_por_asignacion()
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 10
Costo packaging: 28610320.17999999
Costo flete: 154067400
Costo total: 182677720.17999998
Utilización de pallet promedio: 0.8367312371832895
Utilización de caja promedio: 1.189175218430714


In [66]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy5 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy5.resumen_por_asignacion()
solucion_greedy5.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 23
Costo packaging: 29017128.854999997
Costo flete: 135624450
Costo total: 164641578.855
Utilización de pallet promedio: 0.885646681765093
Utilización de caja promedio: 1.2622946456356468


In [67]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy6 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy6.resumen_por_asignacion()
solucion_greedy6.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 25
Costo packaging: 29067442.105000004
Costo flete: 135621450
Costo total: 164688892.10500002
Utilización de pallet promedio: 0.8831659655108768
Utilización de caja promedio: 1.2658219045897858


In [68]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy7 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy7.resumen_por_asignacion()
solucion_greedy7.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 11
Costo packaging: 28679836.24999998
Costo flete: 155208000
Costo total: 183887836.24999997
Utilización de pallet promedio: 0.913412031024089
Utilización de caja promedio: 1.0691205378647197


In [69]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy8 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy8.resumen_por_asignacion()
solucion_greedy8.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 10
Costo packaging: 28610248.029999975
Costo flete: 148891050
Costo total: 177501298.02999997
Utilización de pallet promedio: 0.8598775934557161
Utilización de caja promedio: 1.181297397799954
